# Chapter 7: Deep Learning for Soccer Analytics

This notebook is the canonical Chapter 7 notebook for the companion code.
It follows the main chapter flow through neural-network fundamentals,
PyTorch classification, and goal prediction with multi-task learning.

For the separate TensorFlow companion, see
`notebooks/chapter-7/chapter-7-tensorflow-companion.ipynb`.
        


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

## Why Deep Learning for Soccer? Beyond Traditional Models

### The Intuition: A Team of Players

Imagine a soccer team on the pitch. Each player has a role:
- **Defenders** block attacks
- **Midfielders** control the flow
- **Strikers** create scoring opportunities

Together, they process information (passes, positioning, pressing intensity) and turn it into goals.

A **neural network** works similarly. It's a team of interconnected "players" called **neurons**, organized into **layers**. These neurons work together to transform input data (match statistics) into predictions (goals scored, match outcomes).

### Three Types of Layers

1. **Input Layer**: Receives raw data (e.g., soccer statistics), with each neuron representing a specific feature
2. **Hidden Layers**: Process data and identify complex patterns. A network with 2+ hidden layers is called **deep learning**
3. **Output Layer**: Generates the network's predictions (e.g., match results)

### Inside the Model: How Neural Networks Capture Patterns

Just like formations are made up of individual players, layers in neural networks are made up of **neurons**.

### Weights and Biases

The "knowledge" of a neural network is quantified in **weights**. Think of them as the strength of connections between neurons.

**Example**: Teams that take many shots on target typically score more goals. In a neural network, the connection (weight) between a neuron representing "shots on target" and a neuron representing "goals scored" would become strong as the model learns this relationship.

Each neuron also has a **bias** - a personal tendency that shifts its decision-making independent of inputs. Think of it as a player's natural inclination to attack or defend even before receiving the ball.

In [ ]:
# Simple neuron computation
def neuron_output(inputs, weights, bias):
    """
    Calculate the output of a single neuron.
    
    Parameters:
    - inputs: array of input values
    - weights: array of weights (same length as inputs)
    - bias: single bias value
    
    Returns:
    - weighted sum + bias
    """
    return np.dot(inputs, weights) + bias

# Example: Predicting if a team will score based on shots and possession
inputs = np.array([15, 60])  # [shots, possession%]
weights = np.array([0.3, 0.1])  # learned weights
bias = -2.0  # learned bias

output = neuron_output(inputs, weights, bias)
print(f"Inputs: {inputs}")
print(f"Weights: {weights}")
print(f"Bias: {bias}")
print(f"Neuron output (before activation): {output:.3f}")

#### Activation Functions

Weights and biases alone aren't enough. We need to decide when information flows through the network and when it doesn't.

Think of a sloppy pass in soccer: if it's bad enough, the home team intercepts; if it's precise, the ball goes through. That "threshold effect" is what **activation functions** model.

### The Three Most Common Activation Functions

1. **ReLU (Rectified Linear Unit)**
2. **Sigmoid**
3. **Hyperbolic Tangent (tanh)**

##### ReLU (Rectified Linear Unit)

The ReLU function is simple: give an input below zero, and it outputs zero. Above zero, it equals the input value.

$$f(x) = \max(0, x)$$

**Soccer analogy**: Think of it like goal counts - there's no such thing as negative goals, but once the team starts scoring, every additional goal adds up.

**Advantages**: Makes networks efficient (lots of "zeros" where nothing happens) and flexible (many ReLUs stacked together create complex patterns)

**Drawback**: If too many inputs fall below zero, parts of the network "die" and stop learning

In [ ]:
# ReLU activation function
def relu(x):
    return np.maximum(0, x)

# Visualize ReLU
x = np.linspace(-3, 3, 100)
y_relu = relu(x)

plt.figure(figsize=(10, 5))
plt.plot(x, y_relu, linewidth=2, label='ReLU')
plt.axhline(y=0, color='k', linestyle='--', alpha=0.3)
plt.axvline(x=0, color='k', linestyle='--', alpha=0.3)
plt.xlabel('Input (x)', fontsize=12)
plt.ylabel('Output f(x)', fontsize=12)
plt.title('ReLU Activation Function', fontsize=14, fontweight='bold')
plt.legend(fontsize=11)
plt.grid(True, alpha=0.3)
plt.show()

# Example
test_values = np.array([-2, -1, 0, 1, 2])
print(f"Input values: {test_values}")
print(f"ReLU outputs: {relu(test_values)}")

##### Sigmoid Function

The sigmoid function provides smooth, probabilistic outputs between 0 and 1.

$$y = \frac{1}{1 + e^{-x}}$$

**Soccer analogy**: Perfect for outputs we want to read as probabilities - for example, "What's the chance the home team wins?"

**Advantages**: Outputs are always between 0 and 1, making them interpretable as probabilities

**Drawback**: When values get too extreme, the curve flattens and learning slows (the "vanishing gradient problem")

In [ ]:
# Sigmoid activation function
def sigmoid(x):
    return 1 / (1 + np.exp(-x))

# Visualize Sigmoid
y_sigmoid = sigmoid(x)

plt.figure(figsize=(10, 5))
plt.plot(x, y_sigmoid, linewidth=2, label='Sigmoid', color='orange')
plt.axhline(y=0.5, color='r', linestyle='--', alpha=0.3, label='y=0.5')
plt.axvline(x=0, color='k', linestyle='--', alpha=0.3)
plt.xlabel('Input (x)', fontsize=12)
plt.ylabel('Output f(x)', fontsize=12)
plt.title('Sigmoid Activation Function', fontsize=14, fontweight='bold')
plt.legend(fontsize=11)
plt.grid(True, alpha=0.3)
plt.ylim(-0.1, 1.1)
plt.show()

# Example
print(f"Input values: {test_values}")
print(f"Sigmoid outputs: {sigmoid(test_values)}")
print(f"\nNote: All outputs are between 0 and 1 (probabilities)")

##### Hyperbolic Tangent (tanh)

Tanh is similar to sigmoid but ranges from -1 to 1, so it's centered on zero.

$$\tanh(x) = \frac{e^x - e^{-x}}{e^x + e^{-x}}$$

**Advantages**: Can sometimes train faster than sigmoid

**Drawback**: Like sigmoid, it also suffers from vanishing gradients

We won't use it much in this chapter, but it's worth knowing as another option in the toolbox.

In [ ]:
# Tanh activation function
y_tanh = np.tanh(x)

# Visualize all three together
plt.figure(figsize=(12, 6))
plt.plot(x, y_relu, linewidth=2, label='ReLU')
plt.plot(x, y_sigmoid, linewidth=2, label='Sigmoid')
plt.plot(x, y_tanh, linewidth=2, label='Tanh')
plt.axhline(y=0, color='k', linestyle='--', alpha=0.3)
plt.axvline(x=0, color='k', linestyle='--', alpha=0.3)
plt.xlabel('Input (x)', fontsize=12)
plt.ylabel('Output f(x)', fontsize=12)
plt.title('Comparison of Activation Functions', fontsize=14, fontweight='bold')
plt.legend(fontsize=11)
plt.grid(True, alpha=0.3)
plt.show()

#### Forward Propagation

Forward propagation is the process of moving data through a network to generate predictions.

Think of it like a team moving the ball up the field:
- **Passes (weights)** and **positioning (biases)** decide where the play develops
- **Activations** decide whether the ball actually goes through

Here's a minimal two-layer example:

In [ ]:
def forward_propagation(X, weights, biases):
    """
    Perform forward propagation through a two-layer neural network.
    
    Parameters:
    - X: input features
    - weights: list of weight matrices [weights_layer1, weights_layer2]
    - biases: list of bias vectors [biases_layer1, biases_layer2]
    
    Returns:
    - Output probabilities
    """
    # Compute the linear transformation for the hidden layer: Z = XW + b
    z1 = np.dot(X, weights[0]) + biases[0]
    
    # Apply the ReLU activation function to introduce non-linearity
    a1 = relu(z1)
    
    # Compute the linear transformation for the output layer: Z = AW + b
    z2 = np.dot(a1, weights[1]) + biases[1]
    
    # Apply the Sigmoid activation function to produce output probabilities
    a2 = sigmoid(z2)
    
    return a2

# Example: Simple network
np.random.seed(42)
X_example = np.array([[15, 60]])  # [shots, possession%]

# Initialize random weights and biases
weights = [
    np.random.randn(2, 4) * 0.1,  # Input to hidden (2 inputs, 4 hidden neurons)
    np.random.randn(4, 1) * 0.1   # Hidden to output (4 hidden, 1 output)
]
biases = [
    np.zeros(4),  # Hidden layer bias
    np.zeros(1)   # Output layer bias
]

# Forward pass
output = forward_propagation(X_example, weights, biases)
print(f"Input: {X_example}")
print(f"Predicted probability of home win: {output[0][0]:.3f}")

#### Loss Functions

How do we judge whether a model is "playing well"? In soccer, the scoreboard tells us if a team's strategy is working. In machine learning, that scoreboard is the **loss function**: a number that measures how far off the model's predictions are from reality.

A **lower loss** means our predictions are closer to the truth. During training, the network constantly updates its weights to reduce this loss.

### Common Loss Functions for Soccer Tasks

1. **Binary Cross-Entropy (BCE)**: For yes/no tasks like "Will the home team win?" (1 for win, 0 for not win)
2. **Categorical Cross-Entropy**: For multi-class outcomes like predicting win/draw/loss
3. **Mean Squared Error (MSE)**: For continuous values, such as predicting goals scored
4. **Poisson Regression Loss**: For count data, e.g., predicting how many goals a team scores

In [ ]:
# Binary Cross-Entropy Loss
def binary_cross_entropy(y_true, y_pred):
    """
    Calculate binary cross-entropy loss.
    
    Parameters:
    - y_true: actual labels (0 or 1)
    - y_pred: predicted probabilities (between 0 and 1)
    
    Returns:
    - loss value
    """
    epsilon = 1e-15  # Small value to avoid log(0)
    y_pred = np.clip(y_pred, epsilon, 1 - epsilon)
    return -np.mean(y_true * np.log(y_pred) + (1 - y_true) * np.log(1 - y_pred))

# Example
y_true = np.array([1, 0, 1, 1, 0])  # Actual outcomes
y_pred_good = np.array([0.9, 0.1, 0.85, 0.95, 0.05])  # Good predictions
y_pred_bad = np.array([0.5, 0.5, 0.5, 0.5, 0.5])  # Random predictions

loss_good = binary_cross_entropy(y_true, y_pred_good)
loss_bad = binary_cross_entropy(y_true, y_pred_bad)

print(f"True labels: {y_true}")
print(f"\nGood predictions: {y_pred_good}")
print(f"Loss (good model): {loss_good:.4f}")
print(f"\nBad predictions: {y_pred_bad}")
print(f"Loss (bad model): {loss_bad:.4f}")
print(f"\nLower loss = better predictions!")

In [ ]:
# Mean Squared Error (for regression tasks)
def mean_squared_error(y_true, y_pred):
    """
    Calculate mean squared error.
    
    Parameters:
    - y_true: actual values
    - y_pred: predicted values
    
    Returns:
    - MSE value
    """
    return np.mean((y_true - y_pred) ** 2)

# Example: Predicting goals scored
y_true_goals = np.array([2, 1, 3, 0, 2])  # Actual goals
y_pred_goals_good = np.array([2.1, 0.9, 2.8, 0.2, 1.9])  # Good predictions
y_pred_goals_bad = np.array([0, 0, 1, 2, 3])  # Bad predictions

mse_good = mean_squared_error(y_true_goals, y_pred_goals_good)
mse_bad = mean_squared_error(y_true_goals, y_pred_goals_bad)

print(f"True goals: {y_true_goals}")
print(f"\nGood predictions: {y_pred_goals_good}")
print(f"MSE (good model): {mse_good:.4f}")
print(f"\nBad predictions: {y_pred_goals_bad}")
print(f"MSE (bad model): {mse_bad:.4f}")

### Learning from Data: Training, Loss, and Optimization

Before we can train a model to make good predictions, we need to understand how it learns from mistakes.

In soccer terms: it's not enough to just play forward - the team has to review what went wrong after a match, adjust its strategy, and improve for next time.

### Backpropagation: Reviewing the Game Tape

**Backpropagation** is the method a neural network uses to learn from errors. Imagine replaying a failed attack in slow motion: the coach rewinds the video from the missed shot all the way back to the first pass, pointing out where each player could adjust.

Backpropagation takes the prediction error at the output layer and works backwards through the network, showing each weight how much it contributed to the mistake.

The key output is the **gradient**: a measure of how much changing a specific weight would change the loss.

### Gradient Descent: Making Adjustments

Backprop tells each weight how it contributed to an error; **gradient descent** is how the network uses that feedback to improve.

The idea is simple: adjust the weights little by little in the direction that makes the loss smaller.

Think of a coach fine-tuning tactics after a match. If the team pressed too aggressively and got caught on the counter, next time they press a little less. Each game is an "iteration," nudging strategy closer to what works.

In [ ]:
# Simplified gradient descent visualization
def loss_function(w):
    """Simple quadratic loss function for visualization"""
    return (w - 3) ** 2

# Gradient descent
learning_rate = 0.1
w = 0  # Starting weight
history = [w]

for i in range(20):
    # Calculate gradient (derivative of loss)
    gradient = 2 * (w - 3)
    
    # Update weight
    w = w - learning_rate * gradient
    history.append(w)

# Visualize
w_range = np.linspace(-1, 6, 100)
loss_range = loss_function(w_range)

plt.figure(figsize=(12, 5))
plt.plot(w_range, loss_range, linewidth=2, label='Loss Function')
plt.scatter(history, [loss_function(w) for w in history], 
            c=range(len(history)), cmap='viridis', s=50, zorder=5)
plt.xlabel('Weight Value', fontsize=12)
plt.ylabel('Loss', fontsize=12)
plt.title('Gradient Descent: Finding the Optimal Weight', fontsize=14, fontweight='bold')
plt.legend(fontsize=11)
plt.grid(True, alpha=0.3)
plt.colorbar(label='Iteration')
plt.show()

print(f"Starting weight: {history[0]:.3f}")
print(f"Final weight: {history[-1]:.3f}")
print(f"Optimal weight: 3.000")
print(f"\nThe algorithm successfully found the minimum!")

#### Three Styles of Gradient Descent

1. **Batch Gradient Descent**: Uses the entire dataset to make one update. Very stable, but slow if the dataset is huge.

2. **Stochastic Gradient Descent (SGD)**: Updates after each single sample. Faster to react, but the learning path can be noisy.

3. **Mini-Batch Gradient Descent**: A commonly-used balance - updates after a handful of samples at a time, combining stability with speed.

### Neural Network Fundamentals Summary

In this notebook, you learned:

✅ **Neural networks** are composed of layers of neurons that process information

✅ **Weights** represent the strength of connections between neurons

✅ **Biases** allow neurons to activate more flexibly

✅ **Activation functions** (ReLU, Sigmoid, Tanh) determine when information flows through the network

✅ **Forward propagation** is the process of moving data through the network to make predictions

✅ **Loss functions** measure how far predictions are from reality

✅ **Backpropagation** calculates how each weight contributed to the error

✅ **Gradient descent** uses those gradients to update weights and improve the model

### Next Steps

In the next notebook, we'll put these concepts into practice by building our first neural network in PyTorch to predict home team wins!

## Building Your First Soccer Prediction Model Using PyTorch

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

# Set random seeds for reproducibility
np.random.seed(42)
torch.manual_seed(42)

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")

### Introduction to PyTorch

**PyTorch** is a powerful open-source framework designed for building and training neural networks. It's known for:
- User-friendly interface
- Robust GPU acceleration
- Extensive library ecosystem
- Dynamic computation graphs

In this notebook, we'll build practical soccer predictors with increasing difficulty:
1. **Task 1 (Binary)**: Will the home team win?
2. **Task 2 (Multi-class)**: Win / Draw / Loss prediction

Let's start with the simplest task and work our way up!

### Binary Outcomes: Will the Home Team Win?

For this tutorial, we'll create realistic soccer match data. In practice, you would load this from a CSV file or API.

In [ ]:
# Generate simulated match data
np.random.seed(42)
n_matches = 1000

# Features: shots, possession%, xG (expected goals), corners, fouls
shots_home = np.random.poisson(12, n_matches)
possession_home = np.random.normal(50, 15, n_matches)
xg_home = np.random.gamma(2, 0.6, n_matches)
corners_home = np.random.poisson(5, n_matches)
fouls_home = np.random.poisson(10, n_matches)

shots_away = np.random.poisson(10, n_matches)
possession_away = 100 - possession_home
xg_away = np.random.gamma(1.5, 0.6, n_matches)
corners_away = np.random.poisson(4, n_matches)
fouls_away = np.random.poisson(11, n_matches)

# Create DataFrame
df = pd.DataFrame({
    'shots_home': shots_home,
    'possession_home': possession_home,
    'xg_home': xg_home,
    'corners_home': corners_home,
    'fouls_home': fouls_home,
    'shots_away': shots_away,
    'possession_away': possession_away,
    'xg_away': xg_away,
    'corners_away': corners_away,
    'fouls_away': fouls_away
})

# Generate outcomes based on features (with some randomness)
win_probability = 1 / (1 + np.exp(-(0.3 * (df['xg_home'] - df['xg_away']) + 
                                     0.02 * (df['shots_home'] - df['shots_away']) +
                                     0.01 * (df['possession_home'] - 50))))

df['home_win'] = (np.random.random(n_matches) < win_probability).astype(int)

# Create full-time result (FTR)
def generate_ftr(row):
    score_diff = row['xg_home'] - row['xg_away'] + np.random.normal(0, 0.5)
    if score_diff > 0.5:
        return 'H'  # Home win
    elif score_diff < -0.5:
        return 'A'  # Away win
    else:
        return 'D'  # Draw

df['FTR'] = df.apply(generate_ftr, axis=1)

print(f"Dataset shape: {df.shape}")
print(f"\nFirst few rows:")
print(df.head())
print(f"\nHome win rate: {df['home_win'].mean():.2%}")
print(f"\nFTR distribution:")
print(df['FTR'].value_counts(normalize=True))

#### Data Preparation

Before we can train a model, we need to put our data into a format PyTorch understands.

PyTorch provides a handy tool called **DataLoader**, which takes in tensors and produces mini-batches for training. A mini-batch is just a small slice of the data - rather than feeding the entire season of matches into the model at once, we present it with a handful of matches at a time.

In [ ]:
# Select features for binary classification
feature_cols = ['shots_home', 'possession_home', 'xg_home', 'corners_home', 'fouls_home',
                'shots_away', 'possession_away', 'xg_away', 'corners_away', 'fouls_away']

X = df[feature_cols].values
y = df['home_win'].values

# Standardize features (important for neural networks)
scaler = StandardScaler()
X = scaler.fit_transform(X)

print(f"Features shape: {X.shape}")
print(f"Target shape: {y.shape}")
print(f"\nFeature statistics after scaling:")
print(f"Mean: {X.mean(axis=0)}")
print(f"Std: {X.std(axis=0)}")

In [ ]:
# Convert to PyTorch tensors
X_tensor = torch.tensor(X, dtype=torch.float32)
y_tensor = torch.tensor(y, dtype=torch.float32)  # float targets for BCE

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X_tensor, y_tensor, test_size=0.2, random_state=42
)

# Create DataLoaders (mini-batches)
train_data = TensorDataset(X_train, y_train)
test_data = TensorDataset(X_test, y_test)

train_loader = DataLoader(train_data, batch_size=32, shuffle=True)
test_loader = DataLoader(test_data, batch_size=32, shuffle=False)

print(f"Training set: {len(X_train)} matches")
print(f"Test set: {len(X_test)} matches")
print(f"\nBatch size: 32")
print(f"Number of training batches: {len(train_loader)}")
print(f"Number of test batches: {len(test_loader)}")

##### What Just Happened?

1. **Converted to tensors**: PyTorch's version of arrays
2. **Split the dataset**: 80% training, 20% testing
3. **Created DataLoaders**: Will give us shuffled batches of size 32 during training

Shuffling ensures the model doesn't learn anything artificial from the order of matches. Just as you wouldn't want a team to practice only in the order of its fixtures, you want them to mix drills so they're ready for any opponent.

#### Model Architecture

Now we define the neural network itself. For this task, we'll use a simple **Multi-Layer Perceptron (MLP)**:
- One hidden layer with 16 neurons
- ReLU activation
- Single output neuron for win probability

In [ ]:
class BinaryMLP(nn.Module):
    def __init__(self, input_size, hidden_size=16):
        super().__init__()
        self.fc1 = nn.Linear(input_size, hidden_size)
        self.relu = nn.ReLU()
        self.fc2 = nn.Linear(hidden_size, 1)  # one output neuron
        
    def forward(self, x):
        x = self.fc1(x)
        x = self.relu(x)
        x = self.fc2(x)
        return x  # return logits (will apply sigmoid in loss function)

# Create model
input_size = X_train.shape[1]
model_binary = BinaryMLP(input_size, hidden_size=16)

# Loss function and optimizer
criterion_binary = nn.BCEWithLogitsLoss()  # stable binary cross-entropy
optimizer_binary = optim.Adam(model_binary.parameters(), lr=1e-3)

print(f"Model architecture:")
print(model_binary)
print(f"\nTotal parameters: {sum(p.numel() for p in model_binary.parameters())}")

##### Understanding the Architecture

- **Input layer**: Matches the number of features (10 in our case)
- **Hidden layer**: 16 neurons that learn tactical patterns (e.g., "dominates possession and has a strong striker")
- **Output layer**: One neuron giving a score (logit) that will be converted to win probability

We use **BCEWithLogitsLoss** (Binary Cross-Entropy with Logits) which combines sigmoid activation and BCE loss for numerical stability.

The **Adam optimizer** is like a flexible coach who adapts strategy from game to game, making updates efficiently.

#### Training

Now comes the learning process. Training is an iterative cycle where the model:
1. Sees batches of matches
2. Makes predictions
3. Compares them against reality
4. Adjusts its weights accordingly

In [ ]:
# Training loop
num_epochs = 100
loss_history = []

for epoch in range(num_epochs):
    model_binary.train()
    running_loss = 0.0
    
    for inputs, targets in train_loader:
        # Zero the gradients
        optimizer_binary.zero_grad()
        
        # Forward pass
        logits = model_binary(inputs).squeeze(1)  # (batch_size,)
        
        # Compute loss
        loss = criterion_binary(logits, targets)
        
        # Backward pass
        loss.backward()
        
        # Update weights
        optimizer_binary.step()
        
        running_loss += loss.item()
    
    # Calculate average loss for this epoch
    avg_loss = running_loss / len(train_loader)
    loss_history.append(avg_loss)
    
    # Print progress every 10 epochs
    if (epoch + 1) % 10 == 0:
        print(f"Epoch {epoch+1:3d}/{num_epochs} - Loss: {avg_loss:.4f}")

print("\nTraining complete!")

##### What Happened During Training?

Each **epoch** is like a season - the team plays through all training matches once.

For every mini-batch:
1. **Forward pass**: Model makes its best guess at who wins
2. **Loss calculation**: Compares guess to actual result
3. **Backpropagation**: Identifies which weights were responsible for mistakes
4. **Weight update**: Optimizer nudges weights toward better predictions

By repeating this 100 times, the model gradually learns patterns that link pre-match features to outcomes.

In [ ]:
# Visualize training progress
plt.figure(figsize=(10, 5))
plt.plot(loss_history, linewidth=2)
plt.xlabel('Epoch', fontsize=12)
plt.ylabel('Loss', fontsize=12)
plt.title('Training Loss Over Time', fontsize=14, fontweight='bold')
plt.grid(True, alpha=0.3)
plt.show()

print(f"Initial loss: {loss_history[0]:.4f}")
print(f"Final loss: {loss_history[-1]:.4f}")
print(f"Improvement: {(1 - loss_history[-1]/loss_history[0])*100:.1f}%")

#### Evaluating the Model

Now let's see how well the model generalizes to unseen matches. This is where the test set comes in.

In [ ]:
# Evaluation
model_binary.eval()
with torch.no_grad():
    logits = model_binary(X_test).squeeze(1)
    probs = torch.sigmoid(logits)  # convert logits to probabilities
    preds = (probs >= 0.5).float()  # threshold at 0.5

# Calculate accuracy
acc = (preds == y_test).float().mean().item()
print(f"Test Accuracy: {acc:.3f} ({acc*100:.1f}%)")

# Detailed classification report
y_test_np = y_test.numpy()
preds_np = preds.numpy()

print("\nClassification Report:")
print(classification_report(y_test_np, preds_np, target_names=['No Win', 'Win']))

# Confusion matrix
cm = confusion_matrix(y_test_np, preds_np)
print("\nConfusion Matrix:")
print(cm)

In [ ]:
# Visualize confusion matrix
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
            xticklabels=['No Win', 'Win'],
            yticklabels=['No Win', 'Win'])
plt.xlabel('Predicted', fontsize=12)
plt.ylabel('Actual', fontsize=12)
plt.title('Confusion Matrix: Home Win Prediction', fontsize=14, fontweight='bold')
plt.show()

##### Understanding the Results

We switch the model to **evaluation mode**, which disables training-specific features like dropout.

The raw outputs are **logits**, so we apply a **sigmoid function** to convert them to probabilities. A value of 0.73 means "the model believes there's a 73% chance the home team wins this match."

We then apply a threshold (0.5) to turn probabilities into binary predictions.

The **accuracy** tells us what fraction of test matches were predicted correctly. This is like asking: out of all the games we didn't let the model see beforehand, how many did it call right?

In [ ]:
# Look at some example predictions
print("Sample predictions:")
print("\nActual | Predicted Prob | Prediction")
print("-" * 40)
for i in range(10):
    actual = int(y_test[i].item())
    prob = probs[i].item()
    pred = int(preds[i].item())
    print(f"{actual:^6} | {prob:^14.3f} | {pred:^10}")

### Match Results: Win, Draw, or Loss

We prepare the data similarly, but the target variable is now categorical. We'll use a **LabelEncoder** to convert H/D/A into numeric class labels 0, 1, and 2.

In [ ]:
# Convert categorical FTR labels to numeric
le = LabelEncoder()
y_multiclass = le.fit_transform(df['FTR'])

print(f"Label encoding:")
for i, label in enumerate(le.classes_):
    print(f"  {label} -> {i}")

print(f"\nClass distribution:")
for i, label in enumerate(le.classes_):
    count = (y_multiclass == i).sum()
    print(f"  {label}: {count} ({count/len(y_multiclass)*100:.1f}%)")

In [ ]:
# Convert to tensors
y_multiclass_tensor = torch.tensor(y_multiclass, dtype=torch.long)  # long type for CrossEntropyLoss

# Train-test split
X_train_mc, X_test_mc, y_train_mc, y_test_mc = train_test_split(
    X_tensor, y_multiclass_tensor, test_size=0.2, random_state=42
)

# Create DataLoaders
train_data_mc = TensorDataset(X_train_mc, y_train_mc)
test_data_mc = TensorDataset(X_test_mc, y_test_mc)

train_loader_mc = DataLoader(train_data_mc, batch_size=32, shuffle=True)
test_loader_mc = DataLoader(test_data_mc, batch_size=32, shuffle=False)

print(f"Training set: {len(X_train_mc)} matches")
print(f"Test set: {len(X_test_mc)} matches")

#### Model Architecture

We now define a network that can output **three scores** instead of one. Each score corresponds to one possible result: home win, draw, or away win.

In [ ]:
class MultiClassMLP(nn.Module):
    def __init__(self, input_size, hidden_size=16, num_classes=3):
        super().__init__()
        self.fc1 = nn.Linear(input_size, hidden_size)
        self.relu = nn.ReLU()
        self.fc2 = nn.Linear(hidden_size, num_classes)  # three outputs
        
    def forward(self, x):
        x = self.fc1(x)
        x = self.relu(x)
        x = self.fc2(x)  # logits for each class
        return x

# Create model
model_multiclass = MultiClassMLP(input_size, hidden_size=16, num_classes=3)

# Loss function and optimizer
criterion_multiclass = nn.CrossEntropyLoss()  # for multi-class problems
optimizer_multiclass = optim.Adam(model_multiclass.parameters(), lr=1e-3)

print(f"Model architecture:")
print(model_multiclass)
print(f"\nTotal parameters: {sum(p.numel() for p in model_multiclass.parameters())}")

##### Key Difference from Binary Classification

The output layer now has **three neurons**, one for each class. The model outputs raw scores called **logits**.

**CrossEntropyLoss** automatically applies softmax to convert these logits into probabilities that sum to one. In soccer terms, the model is learning to distribute its confidence between the three possible outcomes.

#### Training the Model

In [ ]:
# Training loop
num_epochs = 100
loss_history_mc = []

for epoch in range(num_epochs):
    model_multiclass.train()
    running_loss = 0.0
    
    for inputs, labels in train_loader_mc:
        optimizer_multiclass.zero_grad()
        
        # Forward pass
        outputs = model_multiclass(inputs)  # logits for 3 classes
        
        # Compute loss
        loss = criterion_multiclass(outputs, labels)
        
        # Backward pass
        loss.backward()
        
        # Update weights
        optimizer_multiclass.step()
        
        running_loss += loss.item()
    
    avg_loss = running_loss / len(train_loader_mc)
    loss_history_mc.append(avg_loss)
    
    if (epoch + 1) % 10 == 0:
        print(f"Epoch {epoch+1:3d}/{num_epochs} - Loss: {avg_loss:.4f}")

print("\nTraining complete!")

In [ ]:
# Visualize training progress
plt.figure(figsize=(10, 5))
plt.plot(loss_history_mc, linewidth=2, color='green')
plt.xlabel('Epoch', fontsize=12)
plt.ylabel('Loss', fontsize=12)
plt.title('Multi-Class Training Loss Over Time', fontsize=14, fontweight='bold')
plt.grid(True, alpha=0.3)
plt.show()

#### Evaluating the Model

Evaluation looks similar, except now we're dealing with three classes. We'll measure accuracy and look at the confusion matrix to see which results are most often misclassified.

In [ ]:
# Evaluation
model_multiclass.eval()
all_preds = []
all_labels = []

with torch.no_grad():
    for inputs, labels in test_loader_mc:
        outputs = model_multiclass(inputs)
        _, predicted = torch.max(outputs, 1)  # pick class with highest logit
        all_preds.extend(predicted.numpy())
        all_labels.extend(labels.numpy())

all_preds = np.array(all_preds)
all_labels = np.array(all_labels)

# Calculate accuracy
acc_mc = accuracy_score(all_labels, all_preds)
print(f"Test Accuracy: {acc_mc:.3f} ({acc_mc*100:.1f}%)")

# Classification report
print("\nClassification Report:")
print(classification_report(all_labels, all_preds, target_names=le.classes_))

# Confusion matrix
cm_mc = confusion_matrix(all_labels, all_preds)
print("\nConfusion Matrix:")
print(cm_mc)

In [ ]:
# Visualize confusion matrix
plt.figure(figsize=(8, 6))
sns.heatmap(cm_mc, annot=True, fmt='d', cmap='Greens',
            xticklabels=le.classes_,
            yticklabels=le.classes_)
plt.xlabel('Predicted', fontsize=12)
plt.ylabel('Actual', fontsize=12)
plt.title('Confusion Matrix: Match Result Prediction', fontsize=14, fontweight='bold')
plt.show()

##### Interpreting Multi-Class Results

The **confusion matrix** is particularly useful here. It shows:
- **Diagonal**: Correct predictions
- **Off-diagonal**: Misclassifications

For example, you might notice that draws are harder to predict than wins/losses. This is common in soccer - draws often depend on subtle factors that are hard to capture in statistics alone.

In [ ]:
# Look at prediction probabilities
model_multiclass.eval()
with torch.no_grad():
    sample_outputs = model_multiclass(X_test_mc[:10])
    sample_probs = torch.softmax(sample_outputs, dim=1)

print("Sample predictions with probabilities:")
print("\nActual | Predicted | P(H)  | P(D)  | P(A)")
print("-" * 50)
for i in range(10):
    actual = le.classes_[y_test_mc[i]]
    pred = le.classes_[all_preds[i]]
    ph, pd, pa = sample_probs[i].numpy()
    print(f"{actual:^6} | {pred:^9} | {ph:.3f} | {pd:.3f} | {pa:.3f}")

### PyTorch Classification Summary

In this notebook, you learned:

✅ How to prepare soccer data for PyTorch neural networks

✅ How to build and train a **binary classification** model (home win prediction)

✅ How to build and train a **multi-class classification** model (win/draw/loss)

✅ The complete workflow: data → model → training → evaluation

✅ How to interpret model predictions and probabilities

✅ The importance of proper data splitting and evaluation

### Next Steps

In the next notebook, we'll explore **regression tasks** - predicting the actual number of goals scored, and build **multi-output models** that predict both home and away goals simultaneously!

## Goal Prediction: Modeling Scoring Output

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error
from scipy import stats

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

np.random.seed(42)
torch.manual_seed(42)

print(f"PyTorch version: {torch.__version__}")

### Introduction to Regression and Multi-Output Models

In previous notebooks, we predicted **categories** (win/draw/loss). Now we'll predict **counts** - specifically, the number of goals scored.

This is fundamentally different:
- **Classification**: Discrete categories (win, draw, loss)
- **Regression**: Continuous or count values (0, 1, 2, 3... goals)

For goals, we'll use **Poisson regression**, which assumes goals follow a Poisson distribution - perfect for modeling rare events like goals in soccer.

### Generate Simulated Soccer Data with Goals

In [ ]:
# Generate simulated match data with goals
np.random.seed(42)
n_matches = 1000

# Features
shots_home = np.random.poisson(12, n_matches)
possession_home = np.random.normal(50, 15, n_matches)
xg_home = np.random.gamma(2, 0.6, n_matches)
corners_home = np.random.poisson(5, n_matches)
fouls_home = np.random.poisson(10, n_matches)

shots_away = np.random.poisson(10, n_matches)
possession_away = 100 - possession_home
xg_away = np.random.gamma(1.5, 0.6, n_matches)
corners_away = np.random.poisson(4, n_matches)
fouls_away = np.random.poisson(11, n_matches)

# Create DataFrame
df = pd.DataFrame({
    'shots_home': shots_home,
    'possession_home': possession_home,
    'xg_home': xg_home,
    'corners_home': corners_home,
    'fouls_home': fouls_home,
    'shots_away': shots_away,
    'possession_away': possession_away,
    'xg_away': xg_away,
    'corners_away': corners_away,
    'fouls_away': fouls_away
})

# Generate goals based on xG (with Poisson distribution)
df['FTHG'] = np.random.poisson(df['xg_home'])  # Full-Time Home Goals
df['FTAG'] = np.random.poisson(df['xg_away'])  # Full-Time Away Goals

print(f"Dataset shape: {df.shape}")
print(f"\nGoal statistics:")
print(df[['FTHG', 'FTAG']].describe())
print(f"\nAverage home goals: {df['FTHG'].mean():.2f}")
print(f"Average away goals: {df['FTAG'].mean():.2f}")

In [ ]:
# Visualize goal distributions
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Home goals
axes[0].hist(df['FTHG'], bins=range(0, df['FTHG'].max()+2), 
             alpha=0.7, edgecolor='black', color='blue')
axes[0].set_xlabel('Goals', fontsize=12)
axes[0].set_ylabel('Frequency', fontsize=12)
axes[0].set_title('Distribution of Home Goals', fontsize=14, fontweight='bold')
axes[0].grid(True, alpha=0.3)

# Away goals
axes[1].hist(df['FTAG'], bins=range(0, df['FTAG'].max()+2), 
             alpha=0.7, edgecolor='black', color='red')
axes[1].set_xlabel('Goals', fontsize=12)
axes[1].set_ylabel('Frequency', fontsize=12)
axes[1].set_title('Distribution of Away Goals', fontsize=14, fontweight='bold')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("Note: Goals follow a Poisson-like distribution - most matches have 0-3 goals")

### Understanding Poisson Distribution

The Poisson distribution models the probability of a given number of events occurring in a fixed interval.

For goals, it's characterized by one parameter: **λ (lambda)** - the expected number of goals.

If λ = 1.7, the probabilities are:
- P(0 goals) ≈ 18%
- P(1 goal) ≈ 31%
- P(2 goals) ≈ 27%
- P(3 goals) ≈ 15%
- etc.

In [ ]:
# Visualize Poisson distribution
lambda_val = 1.7
x = np.arange(0, 8)
poisson_probs = stats.poisson.pmf(x, lambda_val)

plt.figure(figsize=(10, 5))
plt.bar(x, poisson_probs, alpha=0.7, edgecolor='black')
plt.xlabel('Number of Goals', fontsize=12)
plt.ylabel('Probability', fontsize=12)
plt.title(f'Poisson Distribution (λ = {lambda_val})', fontsize=14, fontweight='bold')
plt.xticks(x)
plt.grid(True, alpha=0.3, axis='y')

for i, prob in enumerate(poisson_probs):
    plt.text(i, prob + 0.01, f'{prob:.2%}', ha='center', fontsize=10)

plt.show()

print(f"Expected value (mean): {lambda_val}")
print(f"Variance: {lambda_val} (same as mean in Poisson distribution)")

### Data Preparation

In [ ]:
# Select features and target
feature_cols = ['shots_home', 'possession_home', 'xg_home', 'corners_home', 'fouls_home',
                'shots_away', 'possession_away', 'xg_away', 'corners_away', 'fouls_away']

X = df[feature_cols].values
y_home_goals = df['FTHG'].values
y_away_goals = df['FTAG'].values

# Standardize features
scaler = StandardScaler()
X = scaler.fit_transform(X)

# Convert to tensors
X_tensor = torch.tensor(X, dtype=torch.float32)
y_home_tensor = torch.tensor(y_home_goals, dtype=torch.float32)
y_away_tensor = torch.tensor(y_away_goals, dtype=torch.float32)

# Train-test split
X_train, X_test, y_home_train, y_home_test = train_test_split(
    X_tensor, y_home_tensor, test_size=0.2, random_state=42
)

# Create DataLoaders
train_data = TensorDataset(X_train, y_home_train)
test_data = TensorDataset(X_test, y_home_test)

train_loader = DataLoader(train_data, batch_size=32, shuffle=True)
test_loader = DataLoader(test_data, batch_size=32, shuffle=False)

print(f"Training set: {len(X_train)} matches")
print(f"Test set: {len(X_test)} matches")
print(f"\nTarget statistics (home goals):")
print(f"  Mean: {y_home_train.mean():.2f}")
print(f"  Std: {y_home_train.std():.2f}")
print(f"  Min: {y_home_train.min():.0f}")
print(f"  Max: {y_home_train.max():.0f}")

### Model Architecture

For count data, we need to ensure the network's output is always **positive** (negative goals make no sense!).

We do this by adding a **Softplus** activation at the end. This guarantees outputs > 0.

The output represents **λ** (lambda) - the expected number of goals.

In [ ]:
class PoissonRegression(nn.Module):
    def __init__(self, input_size):
        super().__init__()
        self.fc1 = nn.Linear(input_size, 32)
        self.relu1 = nn.ReLU()
        self.fc2 = nn.Linear(32, 16)
        self.relu2 = nn.ReLU()
        self.fc3 = nn.Linear(16, 1)
        self.softplus = nn.Softplus()  # ensures positive outputs
        
    def forward(self, x):
        x = self.fc1(x)
        x = self.relu1(x)
        x = self.fc2(x)
        x = self.relu2(x)
        x = self.fc3(x)
        return self.softplus(x)  # λ parameter (expected goals)

# Create model
input_size = X_train.shape[1]
model_poisson = PoissonRegression(input_size)

# Poisson Negative Log-Likelihood Loss
criterion_poisson = nn.PoissonNLLLoss(log_input=False)
optimizer_poisson = optim.Adam(model_poisson.parameters(), lr=0.001)

print(f"Model architecture:")
print(model_poisson)
print(f"\nTotal parameters: {sum(p.numel() for p in model_poisson.parameters())}")

#### Understanding the Architecture

- **Hidden layers with ReLU**: Learn complex combinations (e.g., "high possession + strong attack → more goals")
- **Softplus output**: Outputs positive λ (expected goals)

If λ = 1.8, we interpret that as "on average, the home team scores about 1.8 goals in matches like this."

### Training the Model

We'll implement **early stopping** - halt training if validation performance stops improving.

This is like a coach deciding: "We've trained enough this week - more practice won't make us sharper and might even wear us out."

In [ ]:
# Training loop with early stopping
num_epochs = 200
best_val_loss = float('inf')
patience = 20
patience_counter = 0

train_losses = []
val_losses = []

for epoch in range(num_epochs):
    # Training phase
    model_poisson.train()
    running_loss = 0.0
    
    for inputs, targets in train_loader:
        optimizer_poisson.zero_grad()
        outputs = model_poisson(inputs).squeeze()
        loss = criterion_poisson(outputs, targets)
        loss.backward()
        optimizer_poisson.step()
        running_loss += loss.item()
    
    train_loss = running_loss / len(train_loader)
    train_losses.append(train_loss)
    
    # Validation phase
    model_poisson.eval()
    val_loss = 0.0
    
    with torch.no_grad():
        for inputs, targets in test_loader:
            outputs = model_poisson(inputs).squeeze()
            val_loss += criterion_poisson(outputs, targets).item()
    
    val_loss = val_loss / len(test_loader)
    val_losses.append(val_loss)
    
    # Early stopping check
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        patience_counter = 0
        # Save best model
        best_model_state = model_poisson.state_dict().copy()
    else:
        patience_counter += 1
        if patience_counter >= patience:
            print(f"Early stopping at epoch {epoch+1}")
            # Restore best model
            model_poisson.load_state_dict(best_model_state)
            break
    
    # Logging
    if (epoch + 1) % 20 == 0:
        print(f"Epoch {epoch+1:3d}/{num_epochs} - Train Loss: {train_loss:.4f}, Val Loss: {val_loss:.4f}")

print(f"\nTraining complete! Best validation loss: {best_val_loss:.4f}")

In [ ]:
# Visualize training progress
plt.figure(figsize=(10, 5))
plt.plot(train_losses, label='Training Loss', linewidth=2)
plt.plot(val_losses, label='Validation Loss', linewidth=2)
plt.xlabel('Epoch', fontsize=12)
plt.ylabel('Loss', fontsize=12)
plt.title('Training Progress with Early Stopping', fontsize=14, fontweight='bold')
plt.legend(fontsize=11)
plt.grid(True, alpha=0.3)
plt.show()

### Evaluating the Model

In [ ]:
# Make predictions
model_poisson.eval()
with torch.no_grad():
    y_pred_lambda = model_poisson(X_test).squeeze().numpy()

y_true = y_home_test.numpy()

# Calculate metrics
mse = mean_squared_error(y_true, y_pred_lambda)
mae = mean_absolute_error(y_true, y_pred_lambda)
rmse = np.sqrt(mse)

print(f"Regression Metrics:")
print(f"  Mean Squared Error (MSE): {mse:.4f}")
print(f"  Root Mean Squared Error (RMSE): {rmse:.4f}")
print(f"  Mean Absolute Error (MAE): {mae:.4f}")
print(f"\nInterpretation:")
print(f"  On average, predictions are off by {mae:.2f} goals")

In [ ]:
# Visualize predictions vs actual
plt.figure(figsize=(10, 5))
plt.scatter(y_true, y_pred_lambda, alpha=0.5)
plt.plot([0, y_true.max()], [0, y_true.max()], 'r--', linewidth=2, label='Perfect Prediction')
plt.xlabel('Actual Goals', fontsize=12)
plt.ylabel('Predicted λ (Expected Goals)', fontsize=12)
plt.title('Predicted vs Actual Home Goals', fontsize=14, fontweight='bold')
plt.legend(fontsize=11)
plt.grid(True, alpha=0.3)
plt.show()

#### Interpreting the Predictions

The model outputs **λ** (expected goals). We can use this to calculate probabilities of exact outcomes!

In [ ]:
# Example: Calculate probabilities for a specific match
example_lambda = y_pred_lambda[0]
print(f"Predicted λ: {example_lambda:.2f}")
print(f"\nProbabilities of scoring exactly:")

for goals in range(6):
    prob = stats.poisson.pmf(goals, example_lambda)
    print(f"  {goals} goals: {prob:.2%}")

print(f"\nActual goals scored: {int(y_true[0])}")

In [ ]:
# Show sample predictions
print("Sample Predictions:")
print("\nActual | Predicted λ | Most Likely")
print("-" * 45)
for i in range(15):
    actual = int(y_true[i])
    pred_lambda = y_pred_lambda[i]
    most_likely = int(np.round(pred_lambda))
    print(f"{actual:^6} | {pred_lambda:^11.2f} | {most_likely:^11}")

## Full Match Simulation: From Goals to Outcomes

Predicting home and away goals together lets the model share signal across
related tasks and produce more coherent match-level predictions.
        

### Model Architecture

In [ ]:
class MultiTaskMLP(nn.Module):
    def __init__(self, input_size, hidden_size=64, shared_size=32):
        super().__init__()
        
        # Shared layers (learn general soccer patterns)
        self.shared_layer1 = nn.Linear(input_size, hidden_size)
        self.shared_relu = nn.ReLU()
        self.shared_layer2 = nn.Linear(hidden_size, shared_size)
        
        # Home goals head
        self.home_goals_head = nn.Sequential(
            nn.Linear(shared_size, 32),
            nn.ReLU(),
            nn.Linear(32, 1),
            nn.Softplus()  # ensure non-negative goals
        )
        
        # Away goals head
        self.away_goals_head = nn.Sequential(
            nn.Linear(shared_size, 32),
            nn.ReLU(),
            nn.Linear(32, 1),
            nn.Softplus()  # ensure non-negative goals
        )
    
    def forward(self, x):
        # Shared feature extraction
        x = self.shared_layer1(x)
        x = self.shared_relu(x)
        shared_features = self.shared_layer2(x)
        
        # Task-specific predictions
        home_goals = self.home_goals_head(shared_features)
        away_goals = self.away_goals_head(shared_features)
        
        return home_goals, away_goals

# Create model
model_mtl = MultiTaskMLP(input_size, hidden_size=64, shared_size=32)

print(f"Multi-Task Model Architecture:")
print(model_mtl)
print(f"\nTotal parameters: {sum(p.numel() for p in model_mtl.parameters())}")

### Prepare Data for Multi-Task Learning

In [ ]:
# We need both home and away goals as targets
X_train_mtl, X_test_mtl, y_home_train_mtl, y_home_test_mtl, y_away_train_mtl, y_away_test_mtl = train_test_split(
    X_tensor, y_home_tensor, y_away_tensor, test_size=0.2, random_state=42
)

# Create DataLoaders with both targets
train_data_mtl = TensorDataset(X_train_mtl, y_home_train_mtl, y_away_train_mtl)
test_data_mtl = TensorDataset(X_test_mtl, y_home_test_mtl, y_away_test_mtl)

train_loader_mtl = DataLoader(train_data_mtl, batch_size=32, shuffle=True)
test_loader_mtl = DataLoader(test_data_mtl, batch_size=32, shuffle=False)

print(f"Training set: {len(X_train_mtl)} matches")
print(f"Test set: {len(X_test_mtl)} matches")

### Training the Model

We calculate loss for **both tasks** and combine them:

In [ ]:
# Loss and optimizer
criterion_mtl = nn.PoissonNLLLoss(log_input=False)
optimizer_mtl = optim.Adam(model_mtl.parameters(), lr=0.001)

# Training loop
num_epochs = 150
train_losses_mtl = []

for epoch in range(num_epochs):
    model_mtl.train()
    running_loss = 0.0
    
    for inputs, home_targets, away_targets in train_loader_mtl:
        optimizer_mtl.zero_grad()
        
        # Forward pass - get both predictions
        home_pred, away_pred = model_mtl(inputs)
        home_pred = home_pred.squeeze()
        away_pred = away_pred.squeeze()
        
        # Calculate loss for both tasks
        loss_home = criterion_mtl(home_pred, home_targets)
        loss_away = criterion_mtl(away_pred, away_targets)
        
        # Combined loss (equal weighting)
        total_loss = loss_home + loss_away
        
        # Backward pass
        total_loss.backward()
        optimizer_mtl.step()
        
        running_loss += total_loss.item()
    
    avg_loss = running_loss / len(train_loader_mtl)
    train_losses_mtl.append(avg_loss)
    
    if (epoch + 1) % 20 == 0:
        print(f"Epoch {epoch+1:3d}/{num_epochs} - Total Loss: {avg_loss:.4f}")

print("\nTraining complete!")

In [ ]:
# Visualize training
plt.figure(figsize=(10, 5))
plt.plot(train_losses_mtl, linewidth=2, color='purple')
plt.xlabel('Epoch', fontsize=12)
plt.ylabel('Combined Loss', fontsize=12)
plt.title('Multi-Task Training Progress', fontsize=14, fontweight='bold')
plt.grid(True, alpha=0.3)
plt.show()

### Evaluating the Model

In [ ]:
# Make predictions
model_mtl.eval()
with torch.no_grad():
    home_preds_mtl, away_preds_mtl = model_mtl(X_test_mtl)
    home_preds_mtl = home_preds_mtl.squeeze().numpy()
    away_preds_mtl = away_preds_mtl.squeeze().numpy()

y_home_true = y_home_test_mtl.numpy()
y_away_true = y_away_test_mtl.numpy()

# Calculate metrics for both tasks
print("Multi-Task Model Performance:")
print("\nHome Goals:")
print(f"  MAE: {mean_absolute_error(y_home_true, home_preds_mtl):.4f}")
print(f"  RMSE: {np.sqrt(mean_squared_error(y_home_true, home_preds_mtl)):.4f}")

print("\nAway Goals:")
print(f"  MAE: {mean_absolute_error(y_away_true, away_preds_mtl):.4f}")
print(f"  RMSE: {np.sqrt(mean_squared_error(y_away_true, away_preds_mtl)):.4f}")

In [ ]:
# Visualize predictions for both tasks
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Home goals
axes[0].scatter(y_home_true, home_preds_mtl, alpha=0.5, color='blue')
axes[0].plot([0, y_home_true.max()], [0, y_home_true.max()], 'r--', linewidth=2)
axes[0].set_xlabel('Actual Home Goals', fontsize=12)
axes[0].set_ylabel('Predicted λ (Home)', fontsize=12)
axes[0].set_title('Home Goals Prediction', fontsize=14, fontweight='bold')
axes[0].grid(True, alpha=0.3)

# Away goals
axes[1].scatter(y_away_true, away_preds_mtl, alpha=0.5, color='red')
axes[1].plot([0, y_away_true.max()], [0, y_away_true.max()], 'r--', linewidth=2)
axes[1].set_xlabel('Actual Away Goals', fontsize=12)
axes[1].set_ylabel('Predicted λ (Away)', fontsize=12)
axes[1].set_title('Away Goals Prediction', fontsize=14, fontweight='bold')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Show sample predictions for full matches
print("Sample Match Predictions:")
print("\nActual Score | Predicted Score (λ) | Most Likely Score")
print("-" * 60)
for i in range(15):
    actual_home = int(y_home_true[i])
    actual_away = int(y_away_true[i])
    pred_home = home_preds_mtl[i]
    pred_away = away_preds_mtl[i]
    likely_home = int(np.round(pred_home))
    likely_away = int(np.round(pred_away))
    
    print(f"{actual_home:^5}-{actual_away:<5} | {pred_home:^7.2f}-{pred_away:<7.2f} | {likely_home:^7}-{likely_away:<7}")

### Goal Modeling Summary

In this notebook, you learned:

✅ **Regression with neural networks** for predicting continuous values

✅ **Poisson regression** for modeling count data (goals)

✅ How to use **Softplus activation** to ensure positive outputs

✅ **Early stopping** to prevent overfitting

✅ **Multi-task learning (MTL)** to predict multiple related outcomes

✅ How to build models with **shared layers** and **task-specific heads**

✅ The advantages of MTL: efficiency, accuracy, and robustness

### Key Takeaways

- **Poisson regression** is ideal for rare events like goals
- **λ (lambda)** represents expected goals and can be used to calculate probabilities
- **Multi-task learning** leverages shared patterns across related tasks
- Predicting both teams' goals together is more effective than separate models

## Practice Exercises

### Fundamentals Exercises

1. **Experiment with activation functions**: Create a neuron with different activation functions and observe how the output changes with different inputs.

2. **Build a 3-layer network**: Extend the `forward_propagation` function to include an additional hidden layer.

3. **Loss comparison**: Create predictions for a classification task and calculate both BCE and MSE. Which one is more appropriate and why?

4. **Gradient descent tuning**: Modify the learning rate in the gradient descent example. What happens with very large (e.g., 0.9) or very small (e.g., 0.01) learning rates?

5. **Soccer application**: Think of a soccer prediction task you'd like to solve. What would be your input features? What activation function would you use in the output layer? What loss function would be appropriate?

### PyTorch Exercises

1. **Add more features**: Include additional match statistics (yellow cards, offsides, etc.) and see if accuracy improves.

2. **Experiment with architecture**: Try different numbers of hidden neurons (8, 32, 64) or add a second hidden layer.

3. **Tune hyperparameters**: Adjust the learning rate (try 0.01, 0.001, 0.0001) and batch size (16, 64, 128).

4. **Class imbalance**: If one class dominates, try using weighted loss functions or oversampling techniques.

5. **Probability calibration**: Instead of using 0.5 as the threshold, find the optimal threshold that maximizes accuracy or F1-score.

6. **Feature importance**: Which features are most important? Try training models with different feature subsets.

7. **Early stopping**: Implement early stopping to prevent overfitting by monitoring validation loss.

### Goal Modeling Exercises

1. **Compare single-task vs multi-task**: Train separate models for home and away goals, then compare performance to the MTL model.

2. **Weighted loss**: Experiment with different weights for home vs away loss (e.g., `0.6 * loss_home + 0.4 * loss_away`).

3. **Deeper shared layers**: Add more shared layers to the MTL model. Does it improve performance?

4. **Match outcome from goals**: Use the predicted goals to calculate win/draw/loss probabilities.

5. **Total goals**: Modify the model to predict total goals (home + away) as a third task.

6. **Confidence intervals**: Use the Poisson distribution to calculate confidence intervals for goal predictions.

7. **Feature analysis**: Which features are most important for predicting goals? Try removing features one at a time.

## TensorFlow Companion

This canonical chapter notebook keeps the main Chapter 7 walkthrough in one
place. The separate TensorFlow implementation remains available in
`notebooks/chapter-7/chapter-7-tensorflow-companion.ipynb`.
        
